In [14]:
!pip install -q llama-index==0.9.48 sentence-transformers pypdf streamlit pandas

In [15]:
import os
from google.colab import userdata, files
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
files.upload()  # pick your PDF here

Saving Verixa_SRS_v1_1_updatedfinal.pdf to Verixa_SRS_v1_1_updatedfinal (1).pdf


{'Verixa_SRS_v1_1_updatedfinal (1).pdf': b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0 obj\r\n<</Type/Catalog/Pages 2 0 R/Lang(en-US) /StructTreeRoot 188 0 R/MarkInfo<</Marked true>>>>\r\nendobj\r\n2 0 obj\r\n<</Type/Pages/Count 62/Kids[ 3 0 R 13 0 R 15 0 R 17 0 R 19 0 R 38 0 R 45 0 R 48 0 R 50 0 R 53 0 R 55 0 R 62 0 R 64 0 R 66 0 R 70 0 R 74 0 R 78 0 R 82 0 R 85 0 R 87 0 R 89 0 R 93 0 R 95 0 R 97 0 R 99 0 R 101 0 R 104 0 R 106 0 R 108 0 R 110 0 R 112 0 R 114 0 R 117 0 R 119 0 R 121 0 R 123 0 R 125 0 R 127 0 R 129 0 R 131 0 R 133 0 R 135 0 R 137 0 R 139 0 R 141 0 R 143 0 R 145 0 R 147 0 R 149 0 R 151 0 R 153 0 R 155 0 R 157 0 R 159 0 R 161 0 R 164 0 R 167 0 R 171 0 R 175 0 R 179 0 R 182 0 R 185 0 R] >>\r\nendobj\r\n3 0 obj\r\n<</Type/Page/Parent 2 0 R/Resources<</ExtGState<</GS5 5 0 R/GS8 8 0 R>>/Font<</F1 6 0 R/F2 9 0 R/F3 11 0 R>>/ProcSet[/PDF/Text/ImageB/ImageC/ImageI] >>/MediaBox[ 0 0 595.32 841.92] /Contents 4 0 R/Group<</Type/Group/S/Transparency/CS/DeviceRGB>>/Tabs/S/StructParents 0>>\

In [16]:
%%writefile rag_core.py
import os
import re
import time
from statistics import mean

import openai
from llama_index import (
    Document, ServiceContext, SimpleDirectoryReader, StorageContext,
    VectorStoreIndex, load_index_from_storage,
)
from llama_index.indices.postprocessor import SentenceTransformerRerank
from llama_index.llms import OpenAI
from llama_index.llms.base import LLMMetadata
from llama_index.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.query_engine import RetrieverQueryEngine
from llama_index.retrievers import AutoMergingRetriever

MODEL = "openai/gpt-oss-120b"  # generator
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "llama-3.3-70b-versatile")  # evaluator
EMBED = "local:BAAI/bge-small-en-v1.5"
GROQ_URL = "https://api.groq.com/openai/v1"

CONFIGS = {
    "2 layers": [2048, 512],
    "3 layers": [2048, 512, 128],
}

class GroqLLM(OpenAI):
    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(context_window=8192, num_output=512,
                           is_chat_model=True, model_name=MODEL)

def get_llm():
    return GroqLLM(model=MODEL, api_key=os.environ["GROQ_API_KEY"],
                   api_base=GROQ_URL, temperature=0.1)

def load_document(path):
    docs = SimpleDirectoryReader(input_files=[path]).load_data()
    return Document(text="\n\n".join(d.text for d in docs))

def build_index(document, llm, chunk_sizes, save_dir):
    node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=chunk_sizes)
    ctx = ServiceContext.from_defaults(llm=llm, embed_model=EMBED, node_parser=node_parser)

    if os.path.exists(save_dir):
        return load_index_from_storage(
            StorageContext.from_defaults(persist_dir=save_dir), service_context=ctx)

    nodes = node_parser.get_nodes_from_documents([document])
    leaf_nodes = get_leaf_nodes(nodes)
    storage = StorageContext.from_defaults()
    storage.docstore.add_documents(nodes)
    index = VectorStoreIndex(leaf_nodes, storage_context=storage, service_context=ctx)
    index.storage_context.persist(persist_dir=save_dir)
    return index

def build_engine(index, llm, top_k=12, rerank_n=6):
    ctx = ServiceContext.from_defaults(llm=llm, embed_model=EMBED)
    base = index.as_retriever(similarity_top_k=top_k)
    retriever = AutoMergingRetriever(base, index.storage_context, verbose=False)
    rerank = SentenceTransformerRerank(top_n=rerank_n, model="BAAI/bge-reranker-base")
    return RetrieverQueryEngine.from_args(
        retriever, node_postprocessors=[rerank], service_context=ctx)

def build_engines(doc_path, tag="doc", **engine_kwargs):
    llm = get_llm()
    document = load_document(doc_path)
    engines = {}
    for name, sizes in CONFIGS.items():
        save_dir = f"idx_{tag}_{len(sizes)}layers"
        index = build_index(document, llm, sizes, save_dir)
        engines[name] = build_engine(index, llm, **engine_kwargs)
    return engines

def ask(engine, question):
    t = time.time()
    resp = engine.query(question)
    return {
        "answer": str(resp),
        "contexts": [n.node.get_content() for n in resp.source_nodes],
        "seconds": round(time.time() - t, 2),
    }

_client = None

def _judge(prompt, retries=3):
    global _client
    if _client is None:
        _client = openai.OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url=GROQ_URL)
    for i in range(retries):
        try:
            r = _client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0)
            m = re.search(r"\d+(\.\d+)?", r.choices[0].message.content or "")
            return min(float(m.group()) / 10, 1.0) if m else 0.0
        except Exception as e:
            print("JUDGE ERROR:", e)
            time.sleep(2 * (i + 1))
    return 0.0

CTX_PROMPT = """Rate how relevant this context is for answering the question, from 0 to 10.
0 = unrelated, 10 = contains exactly what is needed.
Question: {q}
Context: {c}
Reply with only a number."""

GROUND_PROMPT = """Rate how well the answer is supported by the context, from 0 to 10.
10 = every claim in the answer is backed by the context.
0 = the answer is mostly unsupported or made up.
Context: {c}
Answer: {a}
Reply with only a number."""

ANS_PROMPT = """Rate how well the answer addresses the question, from 0 to 10.
10 = directly and completely answers it. 0 = does not answer it.
Question: {q}
Answer: {a}
Reply with only a number."""

def triad(question, answer, contexts):
    ctx_scores = [_judge(CTX_PROMPT.format(q=question, c=c[:3000])) for c in contexts]
    joined = "\n\n".join(contexts)[:12000]
    return {
        "context_relevance": round(mean(ctx_scores), 2) if ctx_scores else 0.0,
        "groundedness": round(_judge(GROUND_PROMPT.format(c=joined, a=answer)), 2),
        "answer_relevance": round(_judge(ANS_PROMPT.format(q=question, a=answer)), 2),
    }

Overwriting rag_core.py


In [17]:
%%writefile eval_triad.py
import sys
import pandas as pd
from rag_core import ask, build_engines, triad

DOC_PATH = sys.argv[1] if len(sys.argv) > 1 else "your_file.pdf"

QUESTIONS = [
    "What is the tool used to build ai agents in verixa?",
    "What is the tool used to build frontend in verixa?",
    # add 8-10 more
]

engines = build_engines(DOC_PATH, tag="eval")

rows = []
for name, engine in engines.items():
    for q in QUESTIONS:
        out = ask(engine, q)
        print("ANSWER:", out["answer"], "| chunks:", len(out["contexts"]))
        scores = triad(q, out["answer"], out["contexts"])
        rows.append({"layers": name, "question": q, "answer": out["answer"],
                     "n_contexts": len(out["contexts"]),
                     "seconds": out["seconds"], **scores})
        print(name, "|", q[:50], "|", scores)

df = pd.DataFrame(rows)
df.to_csv("results.csv", index=False)
summary = df.groupby("layers")[
    ["context_relevance", "groundedness", "answer_relevance", "seconds"]
].mean().round(2)
print("\n", summary)
summary.to_csv("summary.csv")

Overwriting eval_triad.py


In [18]:
%%writefile app.py
import hashlib
import os
import tempfile

import pandas as pd
import streamlit as st
from rag_core import ask, build_engines, triad

st.set_page_config(page_title="Auto-merging RAG", layout="wide")
st.title("Auto-merging RAG: 2 layers vs 3 layers")

if not os.environ.get("GROQ_API_KEY"):
    st.error("Set GROQ_API_KEY first.")
    st.stop()

@st.cache_resource(show_spinner="Building indexes (first run is slow)...")
def load_engines(path, tag):
    return build_engines(path, tag=tag)

with st.sidebar:
    pdf = st.file_uploader("Upload a PDF", type="pdf")
    run_triad = st.checkbox("Score each answer with RAG triad", value=True)

if pdf is None:
    st.info("Upload a PDF to start.")
    st.stop()

data = pdf.getvalue()
tag = hashlib.md5(data).hexdigest()[:10]
path = os.path.join(tempfile.gettempdir(), f"{tag}.pdf")
with open(path, "wb") as f:
    f.write(data)

engines = load_engines(path, tag)
tab_chat, tab_eval = st.tabs(["Ask", "Eval results"])

with tab_chat:
    if "history" not in st.session_state:
        st.session_state.history = []

    q = st.chat_input("Ask something about the document")
    if q:
        result = {"question": q, "runs": {}}
        with st.spinner("Querying both engines..."):
            for name, engine in engines.items():
                out = ask(engine, q)
                if run_triad:
                    out["scores"] = triad(q, out["answer"], out["contexts"])
                result["runs"][name] = out
        st.session_state.history.insert(0, result)

    for item in st.session_state.history:
        st.markdown(f"### {item['question']}")
        cols = st.columns(len(item["runs"]))
        for col, (name, out) in zip(cols, item["runs"].items()):
            with col:
                st.subheader(name)
                st.write(out["answer"])
                st.caption(f"{out['seconds']}s | {len(out['contexts'])} chunks")
                if "scores" in out:
                    st.dataframe(pd.DataFrame([out["scores"]]),
                                 hide_index=True, use_container_width=True)
                with st.expander("Retrieved context"):
                    for i, c in enumerate(out["contexts"], 1):
                        st.markdown(f"**Chunk {i}**")
                        st.text(c[:1500])
        st.divider()

with tab_eval:
    if os.path.exists("summary.csv"):
        st.dataframe(pd.read_csv("summary.csv"), hide_index=True)
        st.dataframe(pd.read_csv("results.csv"), hide_index=True)
    else:
        st.info("Run eval_triad.py first to see results here.")

Overwriting app.py


In [22]:
import os
os.environ["JUDGE_MODEL"] = "openai/gpt-oss-20b"

In [23]:
!python eval_triad.py "Verixa_SRS_v1_1_updatedfinal.pdf"

/usr/local/lib/python3.13/dist-packages/llama_index/download/module.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Loading weights: 100% 199/199 [00:00<00:00, 3880.13it/s]
Loading weights: 100% 199/199 [00:00<00:00, 7446.60it/s]
Loading weights: 100% 201/201 [00:00<00:00, 3787.39it/s]
Loading weights: 100% 199/199 [00:00<00:00, 4154.91it/s]
Loading weights: 100% 199/199 [00:00<00:00, 3766.89it/s]
Loading weights: 100% 201/201 [00:00<00:00, 3829.53it/s]
ANSWER: Verixa builds its AI agents with Google’s Agent Development Kit (Google ADK). | chunks: 6
2 layers | What is the tool used to build ai agents in verixa | {'context_relevance': 0.5, 'groundedness': 1.0, 'answer_relevance': 0.2}
ANSWER: The frontend is built with React and TypeScript using Vite as the build too

In [24]:
!streamlit run app.py &>/dev/null &
!npx localtunnel --port 8501

# If localtunnel asks for a password, run this in another cell:
# !curl ipv4.icanhazip.com

⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://small-friends-type.loca.lt
^C


In [25]:
!pkill -f streamlit
!pkill -f localtunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.2) ...
Setting up cloudflared (2026.11.2) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [26]:
import os
os.environ["JUDGE_MODEL"] = "openai/gpt-oss-20b"   # keep this, the app uses it too

!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > st.log 2>&1 &
!nohup cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1 &
!sleep 10
!grep -o "https://[a-z0-9-]*\.trycloudflare\.com" cf.log | head -1

https://copyrights-option-wisdom-princess.trycloudflare.com
